In [ ]:
import Pkg; Pkg.activate("../")

In [ ]:
Pkg.develop(path = "/home/isaia/Coding/BallArithmetic/")

In [ ]:
using BallArithmetic

In [ ]:
A = BallMatrix(rand(1024, 1024))

In [ ]:
@time BallArithmetic.svdbox(A)

In [ ]:
#import Pkg; Pkg.add("RigorousInvariantMeasures")

In [ ]:
using RigorousInvariantMeasures

In [ ]:
B= RigorousInvariantMeasures.AdjointFourierBasis.FourierAdjoint(128, 32768)

In [ ]:
T(x) = 3.83*x*(1-x)

In [ ]:
#NK = RigorousInvariantMeasures.GaussianNoise(B, 0.1)

In [ ]:
P = RigorousInvariantMeasures.AdjointFourierBasis.assemble_standard(B, T; ϵ = 0.000000001, max_iter = 100)

In [ ]:
#Pkg.add("IntervalArithmetic")

In [ ]:
using IntervalArithmetic
midI = IntervalArithmetic.mid
radI = IntervalArithmetic.radius

In [ ]:
midP = midI.(real.(P))+im*midI.(imag.(P)) 

In [ ]:
radP = sqrt.(radI.(real.(P))^2+radI.(imag.(P))^2)

In [ ]:
BallP = BallMatrix(midP, radP)

In [ ]:
using LinearAlgebra

function BallNoiseKernel(B, σ)
    n = (length(B)-1)/2
    z = [ [x for x in 0:n]; [x for x in -n:-1] ]
    
    v = Interval.(z)
    w = [exp(-2*pi^2*σ^2*k^2) for k in v]

    center = Diagonal(midI.(w))
    radius = Diagonal(radI.(w))

    return BallMatrix(center, radius)
end

NK = BallNoiseKernel(B, 0.01)

In [ ]:
Q = NK*BallP

In [ ]:
#Pkg.add("Pseudospectra")
#Pkg.add("Plots")

In [ ]:
#using Pseudospectra, Plots

In [ ]:
#spectralportrait(Q)

In [ ]:
#plot!([cos(t) for t in 0:0.01:2π], [sin(t) for t in 0:0.01:2π], label = "")

In [ ]:
savefig("pseudospectra.pdf")

In [ ]:
NK.NK[128, 128]

In [ ]:
Z = Q

In [ ]:
using BallArithmetic

In [ ]:
#A = rand(16, 16)

In [ ]:
enc = BallArithmetic.compute_enclosure(BallMatrix(Q), 0.5, 1.1, 0.001, max_steps = 2000, rel_steps = 64)

In [ ]:
#enc = BallArithmetic.compute_enclosure(BallMatrix(Q), 0.4, 1.1, 0.001, max_steps = 2000, rel_steps = 64)

In [ ]:
enc

In [ ]:
enc[1][2]

In [ ]:
using Plots, LinearAlgebra

In [ ]:
pl = plot()

pl = scatter!(pl, eigen(BallArithmetic.mid.(Q)).values, color = :red, label="", markersize = 2.0)

for j in 1:4

for i in 1:length(enc[j][3])-1
    center_x = real(enc[j][3][i])
    center_y = imag(enc[j][3][i])


    radius = 5*abs(enc[j][3][i+1]-enc[j][3][i])/8

    plot!(pl, [center_x+radius*cos(t) for t in 0:0.1:2π], [center_y+radius*sin(t) for t in 0:0.1:2π], color = :blue, label = "")
end

end

circle_1 = [cos(θ) for θ in 0:0.001:2π], [sin(θ) for θ in 0:0.001:2π]
plot!(circle_1[1], circle_1[2], label = "", color = :green)


display(pl)


In [ ]:
savefig("enclosure_noise.pdf")

In [ ]:
function bound_back(ϵ, M, r)
    bepsilon = Ball(ϵ)
    bM = Ball(M)
    br = Ball(r)

    bound1 = (Ball(1.0)+bepsilon)^2*(Ball(1.0)+Ball(2.0)*bepsilon*br)*bM
    
    return bound1/(Ball(1.0)-bepsilon*bound1) 
end

In [ ]:
bound_back(10^(-9), 1120, 1.1)